In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.utils import to_categorical

2025-03-31 09:47:01.359797: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-31 09:47:02.260581: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743414422.530323  448066 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743414422.601332  448066 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1743414423.240006  448066 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disable GPU

import tensorflow as tf
print("Running on CPU:", tf.config.list_physical_devices('GPU'))

Running on CPU: []


2025-03-31 09:48:53.342505: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-03-31 09:48:53.342554: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-03-31 09:48:53.342565: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-03-31 09:48:53.342574: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-03-31 09:48:53.342581: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: hela
2025-03-31 09:48:53.342586: I external/local_xla/xla/stream_executor/cuda/cuda_dia

In [4]:
import numpy as np

# Load using memory-mapped mode (does not load entire file into RAM)
X_labeled = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/X_labeled.npy", mmap_mode='r')
y_labeled = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/y_labeled.npy", mmap_mode='r')
X_unlabeled = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/X_unlabeled.npy", mmap_mode='r')

# Calculate 1/3rd of the dataset size (keeping only 1/3rd)
num_labeled = int(len(X_labeled) * (1/3))
num_unlabeled = int(len(X_unlabeled) * (1/3))

# Select random indices for 1/3rd data
labeled_indices = np.random.choice(len(X_labeled), num_labeled, replace=False)
unlabeled_indices = np.random.choice(len(X_unlabeled), num_unlabeled, replace=False)

# Reduce dataset size
X_labeled_small = X_labeled[labeled_indices]
y_labeled_small = y_labeled[labeled_indices]
X_unlabeled_small = X_unlabeled[unlabeled_indices]

# Verify new shapes
print("Reduced X_labeled shape:", X_labeled_small.shape)
print("Reduced y_labeled shape:", y_labeled_small.shape)
print("Reduced X_unlabeled shape:", X_unlabeled_small.shape)


Reduced X_labeled shape: (4051, 224, 224, 3)
Reduced y_labeled shape: (4051, 3)
Reduced X_unlabeled shape: (4051, 224, 224, 3)


In [3]:
X_labeled_small = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/X_labeled_small.npy",mmap_mode='r')
y_labeled_small = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/y_labeled_small.npy",mmap_mode='r')
X_unlabeled_small = np.load("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/Task_1_scene_level/X_unlabeled_small.npy",mmap_mode='r')

In [ ]:
# np.save("X_labeled_small.npy", X_labeled_small)
# np.save("y_labeled_small.npy", y_labeled_small)
# np.save("X_unlabeled_small.npy", X_unlabeled_small)

In [4]:
model = load_model("/home/sombit-ng-nonadm/vaibhav_intern_pg/CS230Project/vgg16_modelScene_final.h5")


In [5]:
# Get predictions
predictions = model.predict(X_unlabeled_small)


print("First 5 predictions (raw probabilities):")
print(predictions[:5])

# Convert to pseudo-labels
pseudo_labels = np.argmax(predictions, axis=1)

# Display first 5 pseudo-labels
print("First 5 pseudo-labels (predicted class indices):")
print(pseudo_labels[:5])


127/127 ━━━━━━━━━━━━━━━━━━━━ 69s 543ms/step
First 5 predictions (raw probabilities):
[[9.4856596e-01 5.1433999e-02 1.9312057e-16]
 [1.5866405e-18 1.0000000e+00 0.0000000e+00]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]
 [2.7916213e-08 1.0000000e+00 1.8537367e-14]]
First 5 pseudo-labels (predicted class indices):
[0 1 2 2 1]


In [6]:
print("First 5 predictions (raw probabilities):")
print(predictions[1000:1500])

# Convert to pseudo-labels
pseudo_labels = np.argmax(predictions, axis=1)

# Display first 5 pseudo-labels
print("First 5 pseudo-labels (predicted class indices):")
print(pseudo_labels[1000:1500])


First 5 predictions (raw probabilities):
[[4.3110785e-25 0.0000000e+00 1.0000000e+00]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]
 ...
 [1.7455217e-21 1.0000000e+00 9.8696582e-26]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]
 [0.0000000e+00 0.0000000e+00 1.0000000e+00]]
First 5 pseudo-labels (predicted class indices):
[2 2 2 2 2 2 1 1 2 1 0 0 0 1 2 2 2 2 2 1 2 2 2 2 0 1 2 2 0 2 1 1 2 2 1 2 1
 2 1 2 2 0 2 0 1 1 2 2 2 2 1 2 2 1 2 2 2 2 2 1 2 2 0 2 2 2 2 2 2 2 1 2 1 2
 1 2 1 2 1 2 2 2 2 2 0 1 0 2 1 2 0 2 1 2 1 2 2 2 1 2 2 2 1 2 2 2 2 2 1 1 0
 2 1 0 2 2 2 0 2 2 2 2 2 2 1 1 2 0 2 2 1 1 2 0 2 2 2 0 2 2 2 2 2 2 2 1 0 1
 1 2 2 2 0 0 0 0 1 0 2 0 0 2 0 0 2 0 1 0 2 2 0 2 0 1 1 0 0 0 2 0 1 2 2 2 2
 2 2 2 2 2 2 1 2 2 2 1 0 2 0 0 1 0 1 2 0 2 2 2 2 0 1 2 2 2 2 1 2 2 2 2 0 2
 2 2 2 1 2 2 1 1 1 2 1 0 2 1 0 1 1 1 2 1 1 1 2 0 2 1 2 2 1 2 1 1 2 1 2 2 2
 2 2 2 2 0 1 2 2 2 1 2 2 2 2 0 1 1 2 2 2 2 2 2 2 2 2 1 1 1 2 0 1 0 1 2 2 0
 1 2 1 0 1 2 2 0 2 1 0 1 2 2 1 1 2

In [7]:

# Combine datasets
X_combined_small = np.concatenate((X_labeled_small, X_unlabeled_small), axis=0)
# Y_combined = np.concatenate((y_labeled_one_hot, pseudo_labels_one_hot), axis=0)

# Check shapes
print("Shape of X_combined:", X_combined_small.shape)

Shape of X_combined: (8102, 224, 224, 3)


In [8]:
print("Shape of X_combined:", X_combined_small.shape)
print("ylabelshape",y_labeled_small.shape)
print("pseudo_labels shape",pseudo_labels.shape)

Shape of X_combined: (8102, 224, 224, 3)
ylabelshape (4051, 3)
pseudo_labels shape (4051,)


In [9]:
import numpy as np
from tensorflow.keras.utils import to_categorical

# Ensure pseudo_labels is 1D before one-hot encoding
pseudo_labels = np.array(pseudo_labels).reshape(-1)  # Shape (M,)

# Manually set the number of classes
num_classes = 3  # Since it's a 3-class problem

# Convert only pseudo_labels to one-hot encoding
pseudo_labels_one_hot = to_categorical(pseudo_labels, num_classes=num_classes)  # Shape (M, num_classes)

# Check the shapes
print("y_labeled shape (already one-hot):", y_labeled_small.shape)  # Expected: (12154, 3)
print("pseudo_labels_one_hot shape:", pseudo_labels_one_hot.shape)  # Expected: (12155, 3)

# Combine datasets
Y_combined_small = np.concatenate((y_labeled_small, pseudo_labels_one_hot), axis=0)

print("Shape of X_combined:", X_combined_small.shape)  # Expected: (24309, 224, 224, 3)
print("Shape of Y_combined:", Y_combined_small.shape)  # Expected: (24309, 3)

y_labeled shape (already one-hot): (4051, 3)
pseudo_labels_one_hot shape: (4051, 3)
Shape of X_combined: (8102, 224, 224, 3)
Shape of Y_combined: (8102, 3)


In [10]:
from tensorflow.keras.losses import CategoricalCrossentropy

# Hyperparameters
T1 = 5     # Start pseudo-label weighting at epoch 5
T2 = 10    # Full pseudo-label weight applied at epoch 10
alpha_f = 3.0  # Final weight

def combined_loss(y_true, y_pred, epoch):
    """
    Custom loss function with gradually increasing pseudo-label weight.
    """
    labeled_mask = tf.cast(tf.reduce_max(y_true, axis=-1) > 0, tf.float32)  # 1 for labeled data, 0 for pseudo-labels
    unlabeled_mask = 1 - labeled_mask  # 1 for pseudo-labels, 0 for labeled data

    alpha_t = tf.cond(
        tf.less(epoch, T1),
        lambda: 0.0,
        lambda: tf.cond(
            tf.less(epoch, T2),
            lambda: ((epoch - T1) / (T2 - T1)) * alpha_f,
            lambda: alpha_f
        )
    )

    # Compute losses
    loss_labeled = CategoricalCrossentropy()(y_true, y_pred) * labeled_mask
    loss_unlabeled = CategoricalCrossentropy()(y_true, y_pred) * unlabeled_mask * alpha_t


    return tf.reduce_mean(loss_labeled + loss_unlabeled)


In [12]:
import numpy as np
import tensorflow as tf
import time
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import backend as K

class CustomLossCallback(tf.keras.callbacks.Callback):
    """ Updates the loss function dynamically based on the epoch. """
    def on_epoch_begin(self, epoch, logs=None):
        global custom_loss
        custom_loss = lambda y_true, y_pred: combined_loss(y_true, y_pred, epoch)

class EpochTimeCallback(tf.keras.callbacks.Callback):
    """ Measures and displays the time taken for each epoch. """
    def on_epoch_begin(self, epoch, logs=None):
        self.start_time = time.time()
        print(f"\nEpoch {epoch + 1}/{self.params['epochs']} started...")

    def on_epoch_end(self, epoch, logs=None):
        end_time = time.time()
        elapsed_time = end_time - self.start_time
        print(f"Epoch {epoch + 1} completed in {elapsed_time:.2f} seconds.")

def f1_score(y_true, y_pred):
    """ Computes the F1-score as a metric. """
    y_pred_binary = K.round(y_pred)  # Convert probabilities to binary 0/1
    tp = K.sum(K.cast(y_true * y_pred_binary, 'float'), axis=0)
    fp = K.sum(K.cast((1 - y_true) * y_pred_binary, 'float'), axis=0)
    fn = K.sum(K.cast(y_true * (1 - y_pred_binary), 'float'), axis=0)
    
    precision = tp / (tp + fp + K.epsilon())
    recall = tp / (tp + fn + K.epsilon())
    
    f1 = 2 * precision * recall / (precision + recall + K.epsilon())
    return K.mean(f1)

def confidence_score(y_true, y_pred):
    """ Computes the confidence score as the max softmax probability. """
    return K.mean(K.max(y_pred, axis=-1))

# Clone and recompile the model
new_model = tf.keras.models.clone_model(model)
new_model.set_weights(model.get_weights())

# Compile with custom metrics
new_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=lambda y_true, y_pred: combined_loss(y_true, y_pred, 0),  # Initial loss
    metrics=["accuracy", f1_score, confidence_score]
)

# Train
epochs = 15
batch_size = 32
history = new_model.fit(
    X_combined_small, Y_combined_small,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.1,
    callbacks=[CustomLossCallback(), EpochTimeCallback()]
)

# Save model
new_model.save("retrained_model_with_f1.h5")



Epoch 1/15 started...
Epoch 1/15
228/228 ━━━━━━━━━━━━━━━━━━━━ 0s 638ms/step - accuracy: 0.7969 - confidence_score: 0.9878 - f1_score: 0.7681 - loss: 6.2508Epoch 1 completed in 171.34 seconds.
228/228 ━━━━━━━━━━━━━━━━━━━━ 171s 712ms/step - accuracy: 0.7969 - confidence_score: 0.9878 - f1_score: 0.7681 - loss: 6.2455 - val_accuracy: 0.8545 - val_confidence_score: 0.9856 - val_f1_score: 0.8012 - val_loss: 1.3979

Epoch 2/15 started...
Epoch 2/15
228/228 ━━━━━━━━━━━━━━━━━━━━ 0s 585ms/step - accuracy: 0.8041 - confidence_score: 0.9783 - f1_score: 0.7779 - loss: 2.8845Epoch 2 completed in 147.86 seconds.
228/228 ━━━━━━━━━━━━━━━━━━━━ 148s 648ms/step - accuracy: 0.8040 - confidence_score: 0.9783 - f1_score: 0.7779 - loss: 2.8833 - val_accuracy: 0.8163 - val_confidence_score: 0.9755 - val_f1_score: 0.7654 - val_loss: 1.4361

Epoch 3/15 started...
Epoch 3/15
228/228 ━━━━━━━━━━━━━━━━━━━━ 0s 557ms/step - accuracy: 0.8068 - confidence_score: 0.9657 - f1_score: 0.7790 - loss: 1.5886Epoch 3 complete